In [ ]:
!pip install seqeval pysbd optuna

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

import evaluate
metric = evaluate.load("seqeval")

In [ ]:
model_name = "CLTL/MedRoBERTa.nl"
train_file = "medmentions_dutch.csv"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, add_prefix_space=True)
data_collator = DataCollatorForTokenClassification(tokenizer)

max_length = 512

In [ ]:
def normalize_bio_lst(labels):
    fixed = []
    prev = "O"
    for lab in labels:
        if lab == "I" and prev == "O":
            lab = "B"
        fixed.append(lab)
        prev = lab
    return fixed

def tokenize_and_align_labels(examples, max_length):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens
            if word_idx is None:
                label_ids.append(-100)

            # First token of a word
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])

            # Other subtokens of same word
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
df = pd.read_csv(train_file)
df["tokens"] = df["sentence"].apply(lambda x: x.split())
df["labels"] = df["word_labels"].apply(lambda x: [s.strip() for s in x.split(",")])
df["labels"] = df["labels"].apply(normalize_bio_lst)

label_names = ["O", "B", "I"]
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

In [ ]:
df["has_entity"] = df["labels"].apply(lambda labs: any(l != "O" for l in labs))
df = pd.concat([df, df[df["has_entity"]]], ignore_index=True)

train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["has_entity"]
)

train_dataset = Dataset.from_pandas(train_df[["tokens", "labels"]])
eval_dataset= Dataset.from_pandas(test_df[["tokens", "labels"]])

In [ ]:
train_dataset = train_dataset.map(lambda x: tokenize_and_align_labels(x, max_length), batched=True, remove_columns=["tokens", "labels"]).set_format(type="torch")
eval_dataset = eval_dataset.map(lambda x: tokenize_and_align_labels(x, max_length), batched=True, remove_columns=["tokens", "labels"]).set_format(type="torch")

args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=0.0001,
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=42,
)

model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_names),
        label2id=label2id,
        id2label=id2label,
    )
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
output_dir = "ner_model"
os.makedirs(output_dir, exist_ok=True)
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)